In [ ]:
import sys
print("Executing Python from:", sys.executable)
try:
    import numpy
    print("NumPy successfully loaded from:", numpy.__file__)
except ImportError as e:
    print("NumPy import failed!")

print("\nCurrent sys.path:")
for path in sys.path:
    print(" -", path)

In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

print("Loading")
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")


from analysis_village.cc1pi.var_configs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from cols_to_keep import *

from analysis_village.cc1pi.TLExtensionMethod.GaussianFactorFittingUtils import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from makedf.makedf import *
from pyanalib.pandas_helpers import *
from makedf.util import *
from makedf.geniesyst import *
from tqdm import tqdm
from analysis_village.cc1pi.Constants import CTE
from analysis_village.cc1pi.CutMasks import CutMasks
from analysis_village.cc1pi.dEdxCleaning import dEdxCleaning
import lmfit
import uproot as uproot
import os
import ROOT
import subprocess
from ROOT import std
import pickle
from itertools import combinations


# 1. Setup Paths
base_running_path = "."
base_path = base_running_path 

try:
    root_lib_dir = subprocess.check_output(['root-config', '--libdir'], text=True).strip()
except:
    root_lib_dir = "/cvmfs/larsoft.opensciencegrid.org/products/root/v6_28_12/Linux64bit+3.10-2.17-e26-p3915-prof/lib"

# 2. Update Environment
os.environ['LD_LIBRARY_PATH'] = f"{root_lib_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
ROOT.gSystem.AddDynamicPath(root_lib_dir)

# 3. Add Include Path so classes can find each other's headers
ROOT.gInterpreter.AddIncludePath(base_path)

# 4. Explicit Compilation and Loading Function
def load_custom_class(class_name):
    source_file = os.path.join(base_path, f"{class_name}.cpp")
    header_file = os.path.join(base_path, f"{class_name}.h")
    so_file = os.path.join(base_path, f"{class_name}_cpp.so")
    
    # Compile with 'k' (keep), 'O' (optimize), 'f' (force)
    # Force is useful to ensure the symbol table is rebuilt correctly
    status = ROOT.gSystem.CompileMacro(source_file, "kOf")
    
    if status >= 0:
        # CRITICAL: Manually load the shared library to resolve symbols
        if ROOT.gSystem.Load(so_file) < 0:
            print(f"⚠️  Compiled {class_name} but failed to load {so_file}")
            return False
            
        # Declare the header to the interpreter to help dictionary lookup
        ROOT.gInterpreter.Declare(f'#include "{header_file}"')
        print(f"📦 Successfully compiled and linked: {class_name}")
        return True
    else:
        print(f"❌ Failed to compile: {class_name}")
        return False

# 5. Execute Sequence
# We must load PhysdEdx first as Hypfit depends on it
if load_custom_class("PhysdEdx"):
    if load_custom_class("Hypfit"):
        try:
            # Re-declaring just to be safe before instantiation
            ROOT.gInterpreter.ProcessLine(f'#include "{os.path.join(base_path, "Hypfit.h")}"')
            
            # Instantiate
            h_fit = ROOT.Hypfit()
            print("🚀 Success! Hypfit object initialized and linked.")
        except Exception as e:
            print(f"❌ Error during instantiation: {e}")
            # Final fallback: access via C++ global pointer if Python attribute fails
            ROOT.gInterpreter.ProcessLine("Hypfit* h_fit_ptr = new Hypfit();")
            h_fit = ROOT.h_fit_ptr
            print("🚀 Success! Hypfit object initialized via Global Pointer fallback.")

In [ ]:
#Load CV dataframe
keys2load = ["pfp", "hdr", "histpotdf","hit0","hit1","hit2"] ## keys from the configuration file
#bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_stopping.df"
bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_all.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100)
mc_bnb_pfp_df = mc_bnb_df['pfp']
mc_bnb_hit0_df = mc_bnb_df['hit0']
mc_bnb_hit1_df = mc_bnb_df['hit1']
mc_bnb_hit2_df = mc_bnb_df['hit2']
mc_bnb_hdr_df = mc_bnb_df['hdr']

In [ ]:
rename_map = {
    ("pfp", "trk", "len", "", "", ""): "length",
    ("pfp", "trk", "rangeP", "p_muon", "", ""): "range_P",  # Change to 'p_pion' if analyzing pions
    ("pfp", "trk", "truth", "genp", "mag", ""): "true_P",
    ("pfp", "trk", "truth", "pur", "", ""): "purity",
    ("pfp", "trk", "truth", "eff", "", ""): "completeness",
    ("pfp", "trk", "truth", "p", "pdg", ""): "PDG",
    ("pfp", "trk", "truth", "p", "end_process", ""): "end_process",
    ("pfp", "best_plane", "", "", "", ""): "best_plane",
}
if "pion" in bnb_path:
    rename_map = {
        ("pfp", "trk", "len", "", "", ""): "length",
        ("pfp", "trk", "rangeP", "p_pion", "", ""): "range_P",  # Change to 'p_pion' if analyzing pions
        ("pfp", "trk", "truth", "genp", "mag", ""): "true_P",
        ("pfp", "trk", "truth", "pur", "", ""): "purity",
        ("pfp", "trk", "truth", "eff", "", ""): "completeness",
        ("pfp", "trk", "truth", "p", "pdg", ""): "PDG",
        ("pfp", "trk", "truth", "p", "end_process", ""): "end_process",
        ("pfp", "best_plane", "", "", "", ""): "best_plane",
    }

# Direct column reassignment bypasses MultiIndex structural restrictions
mc_bnb_pfp_df.columns = [
    rename_map.get(col, col) for col in mc_bnb_pfp_df.columns
]

# List of renamed columns to keep
target_columns = [
    "length",
    "range_P",
    "true_P",
    "purity",
    "completeness",
    "PDG",
    "end_process",
    "best_plane",
]

# Filter the DataFrame to keep only these target columns
mc_bnb_pfp_df = mc_bnb_pfp_df[
    [c for c in target_columns if c in mc_bnb_pfp_df.columns]
]

In [ ]:
import numpy as np
import pandas as pd

PROCESS_MAP = {
    0: "primary",
    1: "CoupledTransportation",
    2: "FastScintillation",
    3: "Decay",
    4: "anti_neutronInelastic",
    5: "neutronInelastic",
    6: "anti_protonInelastic",
    7: "protonInelastic",
    8: "hadInelastic",
    9: "pipInelastic",
    10: "pimInelastic",
    11: "xipInelastic",
    12: "ximInelastic",
    13: "kaonpInelastic",
    14: "kaonmInelastic",
    15: "sigmapInelastic",
    16: "sigmamInelastic",
    17: "kaon0LInelastic",
    18: "kaon0SInelastic",
    19: "lambdaInelastic",
    20: "anti_lambdaInelastic",
    21: "He3Inelastic",
    22: "ionInelastic",
    23: "xi0Inelastic",
    24: "alphaInelastic",
    25: "tInelastic",
    26: "dInelastic",
    27: "anti_neutronElastic",
    28: "neutronElastic",
    29: "anti_protonElastic",
    30: "protonElastic",
    31: "hadElastic",
    32: "pipElastic",
    33: "pimElastic",
    34: "kaonpElastic",
    35: "kaonmElastic",
    36: "conv",
    37: "phot",
    38: "annihil",
    39: "nCapture",
    40: "nKiller",
    41: "muMinusCaptureAtRest",
    42: "muIoni",
    43: "eBrem",
    44: "CoulombScat",
    45: "hBertiniCaptureAtRest",
    46: "hFritiofCaptureAtRest",
    47: "photonNuclear",
    48: "muonNuclear",
    49: "electronNuclear",
    50: "positronNuclear",
    51: "compt",
    52: "eIoni",
    53: "muBrems",
    54: "hIoni",
    55: "muPairProd",
    56: "hPairProd",
    57: "LArVoxelReadoutScoringProcess",
    58: "ionIoni",
    59: "hBrems",
    60: "Transportation",
    61: "msc",
    62: "StepLimiter",
    63: "LegacyUNKNOWN",
    64: "RadioactiveDecayBase",
    1024: "UNKNOWN",
}


def get_process_name(code):
    # Unpack scalar if code is wrapped in a Series, array, or list
    if isinstance(code, (pd.Series, np.ndarray, list)):
        if len(code) == 0:
            return "UNKNOWN"
        code = code.iloc[0] if isinstance(code, pd.Series) else code[0]

    try:
        if pd.isna(code):
            return "UNKNOWN"
        return PROCESS_MAP.get(int(code), "UNKNOWN")
    except (ValueError, TypeError):
        return "UNKNOWN"

def to_std_vector_double(arr):
    return ROOT.std.vector('double')(np.ascontiguousarray(arr, dtype=np.float64))


def get_track_hits(particle_idx, plane, hit_dfs):
    """Return (rr, dedx, pitch) numpy arrays for one particle on one plane, sorted by rr,
    with non-finite entries dropped -- the pandas equivalent of rr_vec->at(plane) etc.
    Returns None if this particle has no hits on that plane."""
    hit_df = hit_dfs[plane]
    try:
        hits = hit_df.loc[particle_idx]
    except KeyError:
        return None
    if isinstance(hits, pd.Series):
        # a single hit row -- promote to a 1-row frame
        hits = hits.to_frame().T

    hits = hits.sort_values('rr')
    rr    = hits['rr'].to_numpy(dtype=np.float64)
    dedx  = hits['dedx'].to_numpy(dtype=np.float64)
    pitch = hits['pitch'].to_numpy(dtype=np.float64)

    mask = np.isfinite(rr) & np.isfinite(dedx) & np.isfinite(pitch)
    return rr[mask], dedx[mask], pitch[mask]


def compute_reco_KE_P(preset, rr, dedx, pitch, x, length, used_plane, PDG, use_data):
    """Direct port of the big if/else block inside the preset loop of the macro."""
    v_rr    = to_std_vector_double(rr)
    v_dedx  = to_std_vector_double(dedx)
    v_pitch = to_std_vector_double(pitch)
    v_x     = to_std_vector_double(x)
    bad_hits = h_fit.get_hits_to_ignore(v_rr, v_dedx, preset["cleaning_method"])
    key = preset["preset_key"]

    # Pick data- or MC-derived convolution maps once, up front.
    this_conv_tf1_map         = conv_tf1_map
    this_conv_shifted_tf1_map =  conv_shifted_tf1_map
    this_langau_tf1_map = langau_tf1_map

    if "range" in key:
        if len(rr) - 1 == 0:
            reco_KE = h_fit.map_PhysdEdx[PDG].KEFromRangeSpline(length)
        else:
            reco_KE = h_fit.map_PhysdEdx[PDG].KEFromRangeSpline(float(rr[-1]))
    elif "langau" in key:
            reco_KE = h_fit.NormLikelihood_w_convolution(
                v_dedx, v_rr, v_pitch, bad_hits, True, PDG, this_langau_tf1_map[used_plane])
    elif "convolution" in key:
        if "og" in key:
            reco_KE = h_fit.NormLikelihood_w_convolution(
                v_dedx, v_rr, v_pitch, bad_hits, True, PDG, this_conv_tf1_map[used_plane])
        elif "shifted" in key:
            reco_KE = h_fit.NormLikelihood_w_convolution(
                v_dedx, v_rr, v_pitch, bad_hits, True, PDG, this_conv_shifted_tf1_map[used_plane])
        else:
            reco_KE = -999999.0  # unreachable given the preset list above
    else:
        if preset["normalize"]:
            reco_KE = h_fit.NormLikelihood(
                v_dedx, v_rr, v_pitch, bad_hits, preset["include_small_likelihood"], PDG)
        else:
            reco_KE = h_fit.Likelihood(v_dedx, v_rr, v_pitch, bad_hits, PDG)
    reco_P = h_fit.map_PhysdEdx[PDG].KEtoMomentum(reco_KE) / 1000.0
    v_rr.clear(); v_dedx.clear(); v_pitch.clear(); v_x.clear()
    del v_rr, v_dedx, v_pitch, v_x
    return reco_KE, reco_P


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output


def explore_pion_errs_widget(
    pfp_df,
    hit_dfs,
    h_fit,
    pdg=211,
):
    """Interactive Jupyter Event Display using ipywidgets and C++ Hypfit bindings."""
    # 1. Fetch convolution and langau function maps for all planes upfront
    conv_map_og = h_fit.get_conv_function_map(pdg, "none", 1000)
    conv_map_shifted = h_fit.get_conv_function_map(pdg, "shift", 1000)
    langau_map = h_fit.get_langau_map(pdg, 500)

    valid_inelastic = {
        9,
        10,
        9.0,
        10.0,
        "pipInelastic",
        "pimInelastic",
        "pi+Inelastic",
        "pi-Inelastic",
    }

    # 2. Pre-filter candidate event indices
    candidate_indices = []
    print("Pre-scanning dataset for matching pion inelastic events...")

    for particle_idx, pfp_row in pfp_df.iterrows():
        true_P = pfp_row.get("true_P", np.nan)
        if pd.isna(true_P):
            continue

        p_gev = true_P / 1000.0 if true_P > 20.0 else true_P
        if p_gev > 0.5:
            continue

        purity = pfp_row.get("purity", 1.0)
        completeness = pfp_row.get("completeness", 1.0)
        purity = 1.0 if pd.isna(purity) else purity
        completeness = 1.0 if pd.isna(completeness) else completeness
        if purity < 0.8 or completeness < 0.8:
            continue

        raw_end_proc = pfp_row.get("end_process", np.nan)
        if isinstance(raw_end_proc, (pd.Series, np.ndarray, list)):
            raw_end_proc = (
                raw_end_proc.iloc[0]
                if isinstance(raw_end_proc, pd.Series)
                else raw_end_proc[0]
            )

        end_proc_str = get_process_name(raw_end_proc)
        if (
            raw_end_proc not in valid_inelastic
            and end_proc_str not in valid_inelastic
        ):
            continue

        candidate_indices.append(particle_idx)

    print(
        f"Found {len(candidate_indices)} matching events out of {len(pfp_df)} particles."
    )
    if not candidate_indices:
        print(
            "No events passed selection criteria. Please inspect DataFrame values."
        )
        return

    # 3. Create Widget UI Elements
    out = widgets.Output()
    prev_button = widgets.Button(
        description="◀ Previous", button_style="info", icon="arrow-left"
    )
    next_button = widgets.Button(
        description="Next ▶", button_style="primary", icon="arrow-right"
    )
    slider = widgets.IntSlider(
        value=0,
        min=0,
        max=len(candidate_indices) - 1,
        step=1,
        description="Event:",
        continuous_update=False,
    )

    # 4. Event Rendering Callback
    def render_event(change_or_index):
        idx_pos = slider.value
        particle_idx = candidate_indices[idx_pos]
        pfp_row = pfp_df.loc[particle_idx]

        if isinstance(pfp_row, pd.DataFrame):
            pfp_row = pfp_row.iloc[0]

        true_P = pfp_row.get("true_P", np.nan)
        p_gev = true_P / 1000.0 if true_P > 20.0 else true_P

        best_plane_val = pfp_row.get("best_plane", 2)
        best_plane = (
            int(best_plane_val)
            if pd.notna(best_plane_val) and best_plane_val >= 0
            else 2
        )
        used_plane = best_plane if (0 <= best_plane < 3) else 2

        hits = get_track_hits(particle_idx, used_plane, hit_dfs)
        if hits is None:
            with out:
                clear_output(wait=True)
                print(
                    f"No hits found for Particle Index {particle_idx} on plane {used_plane}"
                )
            return

        rr, dedx, pitch = hits
        if len(rr) == 0:
            return

        bad_hits = h_fit.get_hits_to_ignore(rr, dedx, "all")

        good_mask = ~np.isin(np.arange(len(rr)), bad_hits)
        rr_clean = np.array(rr)[good_mask]
        dedx_clean = np.array(dedx)[good_mask]

        # --- Compute Likelihood Profiles for All 3 Options ---
        # 1. OG Conv
        pts_og = h_fit.NormLikelihoodPairVector_w_convolution(
            dedx, rr, pitch, bad_hits, False, pdg, conv_map_og[used_plane]
        )
        rr_og = np.array([lp.rr for lp in pts_og])
        lh_og = np.array([lp.Likelihood for lp in pts_og])

        # 2. Shifted Conv
        pts_shifted = h_fit.NormLikelihoodPairVector_w_convolution(
            dedx, rr, pitch, bad_hits, False, pdg, conv_map_shifted[used_plane]
        )
        rr_shifted = np.array([lp.rr for lp in pts_shifted])
        lh_shifted = np.array([lp.Likelihood for lp in pts_shifted])

        # 3. Langau
        pts_langau = h_fit.NormLikelihoodPairVector_w_convolution(
            dedx, rr, pitch, bad_hits, False, pdg, langau_map[used_plane]
        )
        rr_langau = np.array([lp.rr for lp in pts_langau])
        lh_langau = np.array([lp.Likelihood for lp in pts_langau])

        if len(lh_og) == 0 and len(lh_shifted) == 0 and len(lh_langau) == 0:
            with out:
                clear_output(wait=True)
                print(f"Empty likelihood vectors for particle {particle_idx}")
            return

        # True length reference marker
        ke_true_mev = (
            np.sqrt(p_gev**2 + PION_MASS**2) - PION_MASS
        ) * 1000.0
        range_true = h_fit.map_PhysdEdx[pdg].RangeFromKESpline(ke_true_mev)

        # --- Subplot Construction ---
        with out:
            clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

            # Left Panel: Corrected dE/dx
            ax1.scatter(
                rr_clean,
                dedx_clean,
                color="#1f77b4",
                s=25,
                label="Corrected hits",
                zorder=3,
            )
            ax1.set_xlabel("Residual Range [cm]", fontsize=11)
            ax1.set_ylabel(r"$dE/dx$ [MeV/cm]", fontsize=11)
            ax1.set_title(
                f"Particle Index: {particle_idx} | Plane {used_plane}", fontsize=12
            )
            ax1.grid(True, linestyle=":", color="gray", alpha=0.7)

            # Right Panel: Plot All 3 Likelihood Curves
            all_lh_vals = []

            if len(lh_og) > 0:
                ax2.plot(rr_og, lh_og, color="#1f77b4", linewidth=2, label="OG Conv")
                best_rr_og = rr_og[np.argmin(lh_og)]
                ax2.axvline(best_rr_og, color="#1f77b4", linestyle=":", linewidth=1.5, label="TLE (OG)")
                all_lh_vals.extend(lh_og)

            if len(lh_shifted) > 0:
                ax2.plot(rr_shifted, lh_shifted, color="#d62728", linewidth=2, label="Shifted Conv")
                best_rr_shifted = rr_shifted[np.argmin(lh_shifted)]
                ax2.axvline(best_rr_shifted, color="#d62728", linestyle=":", linewidth=1.5, label="TLE (Shifted)")
                all_lh_vals.extend(lh_shifted)

            if len(lh_langau) > 0:
                ax2.plot(rr_langau, lh_langau, color="#2ca02c", linewidth=2, label="Langau")
                best_rr_langau = rr_langau[np.argmin(lh_langau)]
                ax2.axvline(best_rr_langau, color="#2ca02c", linestyle=":", linewidth=1.5, label="TLE (Langau)")
                all_lh_vals.extend(lh_langau)

            # Plot True Length reference line
            ax2.axvline(
                range_true,
                color="black",
                linestyle="--",
                linewidth=2,
                label="True Length",
            )

            # Set dynamic Y-limits based on all plotted profiles
            if all_lh_vals:
                y_lo, y_hi = np.min(all_lh_vals), np.max(all_lh_vals)
                y_pad = 0.05 * (y_hi - y_lo) if y_hi != y_lo else 1.0
                ax2.set_ylim(y_lo - y_pad, y_hi + y_pad)

            ax2.set_xlabel("Length [cm]", fontsize=11)
            ax2.set_ylabel(r"$-2 \ln \mathcal{L}$", fontsize=11)
            ax2.set_title("Likelihood Profiles Comparison", fontsize=12)
            ax2.grid(True, linestyle=":", color="gray", alpha=0.7)
            ax2.legend(
                loc="upper right",
                frameon=True,
                facecolor="white",
                edgecolor="gray",
                fontsize=9,
            )

            plt.tight_layout()
            display(fig)
            plt.close(fig)

    # 5. Connect UI Controllers
    def on_prev_clicked(b):
        if slider.value > 0:
            slider.value -= 1

    def on_next_clicked(b):
        if slider.value < slider.max:
            slider.value += 1

    prev_button.on_click(on_prev_clicked)
    next_button.on_click(on_next_clicked)
    slider.observe(render_event, names="value")

    # Display control layout and initial plot
    controls = widgets.HBox([prev_button, slider, next_button])
    display(widgets.VBox([controls, out]))
    render_event(None)

In [ ]:
hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]

In [ ]:
# Usage execution:

# Execute Interactive Display Widget
explore_pion_errs_widget(
    mc_bnb_pfp_df,
    hit_dfs,
    h_fit,
    pdg=211
)



In [ ]:
print("DataFrame Columns:", mc_bnb_pfp_df.columns.tolist())
print("\nSample Selection Values:")
sample_cols = [
    c
    for c in ["true_P", "purity", "completeness", "end_process"]
    if c in mc_bnb_pfp_df.columns
]
print(mc_bnb_pfp_df[sample_cols].head(10))
print("\nUnique end_process values:", mc_bnb_pfp_df["end_process"].unique()[:10])